# 3D Kooshball MPnRAGE on the BrainWeb phantom - one scan, per-echo contrast evolution

Same pipeline as `3D_Kooshball_GRE.ipynb` (cyclic 8-channel head coil +
isochromat/Bloch spin simulation + NUFFT preview & iterative NUFFT-SENSE CG),
with the 3D radial (golden-angle) **MPnRAGE** sequence: per segment
`180deg inversion - ti_start - n_readout spoiled GRE echoes (ti_inter apart) - recovery delay`,
repeated `n_segments` times.

**One scan covers every inversion time.** The p-th echo of every segment fires
`TI_eff(p) = ti_start + p (TR + ti_inter)` after that segment's inversion, so
each echo index is a whole 3D volume at a different TI. Reconstructing one image
per echo index (the "frames" of `2D_Radial_MPnRAGE.ipynb`) plots the brain
multiple times and shows the contrast change as the magnetization recovers:
WM nulls first, then GM, then CSF. (Averaging all echoes of a train into one
image - as the per-tissue "null point" version did - washes the null out,
because the effective TI spans seconds and every tissue has recovered by the
later echoes.)

**Which TI nulls a tissue?** The inversion fires every segment (not after full
relaxation), so the magnetization seen by the inversion pulse is the *cyclic*
leftover, not +M0. With echo spacing `P = TR + ti_inter`:

    Mz_0(TI)    = M0 [ 1 - (1 + w) E_TI ],   w = cyclic state before the inversion
    null point  =>  E_TI = 1 / (1 + w),  and at the null w = z* (TI-independent)
    z*          = (1 - E_tot) + E_tot cos(a) kap (1 - lam^(N-1))
    lam = exp(-P/T1) cos(a),  kap = (1 - exp(-P/T1)) / (1 - exp(-P/T1) cos(a))
    E_tot = exp(-(TR+TD)/T1),  E_TD = exp(-TD/T1),  N = n_readout

## Imports and device

All tensors are created on the GPU by default.

In [ ]:
%matplotlib widget

import itertools
import os
import time
from typing import Iterable, Optional, Tuple

import matplotlib.pyplot as plt
import MRzeroCore as mr0
import numpy as np
import pypulseq as pp
import torch
import torch.nn as nn
import torchkbnufft as tkbn
from torch.utils.checkpoint import checkpoint
from tqdm import tqdm

from Modules import default_system, load_brain_phantom
from Modules.CoilSens import cyclic_head_coil
from Modules.fast_iso import isochromat_sim_fast
from Modules.FourierTransform import (
    nonuniform_fourier_transform_adjoint,
    nonuniform_fourier_transform_forward,
)
from Modules.Linops import LinearOperator
from Modules.NumSolvers import conjugate_gradient
from Modules.Plan import KbNufft, KbNufftAdjoint
from Modules.Regularizer import Regularizer
from Modules.Resnet import ResNet

torch.set_default_device("cuda")
device = torch.device("cuda")

## Sequence design

Per segment: 180-degree inversion, `ti_start` delay, then `n_readout` spoiled GRE
echoes separated by `ti_inter`, then a recovery delay. Spoke directions follow a
golden-angle spiral on the sphere over all acquired spokes; the first segment
starts with `n_dummy` dummy shots along x (no ADC).

In [ ]:
def build_mpnrage(
    fov: float = 250e-3,
    n_x: int = 128,
    flip_angle_deg: float = 5.0,
    n_channels: int = 8,
    slice_thickness: float = 3e-3,
    echo_time: float = 3.98e-3,
    n_readout: int = 30,
    n_segments: int = 200,
    n_dummy: int = 20,
    ti_start: float = 20e-3,
    ti_inter: float = 100e-3,
    recovery_delay: float = 500e-3,
    rf_spoiling_increment_deg: float = 117.0,
    write_seq: bool = False,
    filename: str = "MPnRAGE_Kooshball_TI20ms.seq",
) -> Tuple[pp.Sequence, float, float, float]:
    """Build the 3D radial MPnRAGE sequence.

    Parameters
    ----------
    fov : float, optional
        Field of view in m.
    n_x : int, optional
        Number of readout samples per spoke.
    flip_angle_deg : float, optional
        Excitation flip angle in degrees.
    n_channels : int, optional
        Currently unused.
    slice_thickness : float, optional
        Slab thickness in m.
    echo_time : float, optional
        Echo time in s.
    n_readout : int, optional
        Number of echoes (spokes) per inversion segment.
    n_segments : int, optional
        Number of inversion segments.
    n_dummy : int, optional
        Number of dummy shots (no ADC) at the start of the first segment.
    ti_start : float, optional
        Time from the inversion pulse centre to the first excitation in s.
    ti_inter : float, optional
        Recovery delay between consecutive echoes in s.
    recovery_delay : float, optional
        Delay after each echo train in s.
    rf_spoiling_increment_deg : float, optional
        Quadratic RF-spoiling phase increment in degrees.
    write_seq : bool, optional
        Write the sequence to ``filename``.
    filename : str, optional
        Output file name.

    Returns
    -------
    seq : pp.Sequence
        The assembled sequence.
    gre_block_dur : float
        Duration of one GRE block (the TR of one echo) in s.
    delay_ti_inter : float
        Inter-echo delay, rounded up to the gradient raster, in s.
    recovery_delay : float
        Delay after each echo train in s.
    """
    system = default_system()
    seq = pp.Sequence(system)
    delta_k = 1 / fov
    raster = seq.grad_raster_time

    rf_inversion, grad_slice_inversion, _ = pp.make_sinc_pulse(
        flip_angle=np.pi,
        duration=10e-3,
        slice_thickness=slice_thickness,
        apodization=0.5,
        time_bw_product=4,
        system=system,
        return_gz=True,
        use="inversion",
    )

    rf, grad_slice, _ = pp.make_sinc_pulse(
        apodization=0.5,
        duration=2e-3,
        flip_angle=flip_angle_deg * np.pi / 180,
        slice_thickness=slice_thickness,
        system=system,
        time_bw_product=4,
        return_gz=True,
        use="excitation",
    )

    grad_readout_base = pp.make_trapezoid(
        channel="x", flat_area=n_x * delta_k, flat_time=6.4e-3 / 5, system=system
    )
    adc = pp.make_adc(
        num_samples=n_x,
        duration=grad_readout_base.flat_time,
        delay=grad_readout_base.rise_time,
        system=system,
    )
    grad_prephase_base = pp.make_trapezoid(
        channel="x", area=-grad_readout_base.area / 2, duration=2e-3, system=system
    )
    grad_slice_rephase = pp.make_trapezoid(
        channel="z", area=-grad_slice.area / 2, duration=2e-3, system=system
    )
    grad_spoil_base = pp.make_trapezoid(
        channel="x", area=0.5 * n_x * delta_k, system=system
    )
    grad_slice_spoil = pp.make_trapezoid(
        channel="z", area=4 / slice_thickness, system=system
    )

    delay_te = (
        np.ceil(
            (
                echo_time
                - pp.calc_duration(grad_prephase_base)
                - grad_slice.fall_time
                - grad_slice.flat_time / 2
                - pp.calc_duration(grad_readout_base) / 2
            )
            / raster
        )
        * raster
    )
    inversion_duration = pp.calc_duration(rf_inversion, grad_slice_inversion)
    delay_ti_start = (
        np.ceil(
            (ti_start - inversion_duration / 2 - pp.calc_duration(grad_slice) / 2)
            / raster
        )
        * raster
    )
    assert delay_ti_start > 0, "ti_start too short"
    delay_ti_inter = np.ceil(ti_inter / raster) * raster

    gre_block_dur = (
        pp.calc_duration(grad_slice)
        + pp.calc_duration(grad_prephase_base)
        + delay_te
        + pp.calc_duration(grad_readout_base)
        + pp.calc_duration(grad_spoil_base, grad_slice_spoil)
    )
    delay_recovery = np.ceil(recovery_delay / raster) * raster

    n_total_spokes = n_segments * n_readout

    def spoke_direction(idx: int) -> Tuple[float, float, float]:
        """Return the unit direction of acquired spoke ``idx`` (golden-angle spiral).

        Parameters
        ----------
        idx : int
            Index of the acquired spoke.

        Returns
        -------
        Tuple[float, float, float]
            Direction cosines ``(dir_x, dir_y, dir_z)``.
        """
        polar_angle = np.arccos(1 - 2 * (idx + 0.5) / n_total_spokes)
        azimuth_angle = np.pi * (1 + np.sqrt(5)) * idx
        return (
            np.sin(polar_angle) * np.cos(azimuth_angle),
            np.sin(polar_angle) * np.sin(azimuth_angle),
            np.cos(polar_angle),
        )

    rf_phase_deg = 0.0
    rf_inc_deg = 0.0
    spoke_counter = 0

    for segment in range(n_segments):
        seq.add_block(rf_inversion, grad_slice_inversion)
        seq.add_block(pp.make_delay(delay_ti_start))

        n_shots_segment = n_readout + (n_dummy if segment == 0 else 0)
        for shot in range(n_shots_segment):
            is_dummy = (segment == 0) and (shot < n_dummy)

            rf.phase_offset = rf_phase_deg / 180 * np.pi
            adc.phase_offset = rf_phase_deg / 180 * np.pi
            rf_inc_deg = (rf_inc_deg + rf_spoiling_increment_deg) % 360
            rf_phase_deg = (rf_phase_deg + rf_inc_deg) % 360

            if is_dummy:
                dir_x, dir_y, dir_z = 1.0, 0.0, 0.0
            else:
                dir_x, dir_y, dir_z = spoke_direction(spoke_counter)
                spoke_counter += 1

            seq.add_block(rf, grad_slice)
            seq.add_block(
                pp.make_trapezoid(
                    channel="x",
                    area=grad_prephase_base.area * dir_x,
                    duration=2e-3,
                    system=system,
                ),
                pp.make_trapezoid(
                    channel="y",
                    area=grad_prephase_base.area * dir_y,
                    duration=2e-3,
                    system=system,
                ),
                pp.make_trapezoid(
                    channel="z",
                    area=grad_prephase_base.area * dir_z + grad_slice_rephase.area,
                    duration=2e-3,
                    system=system,
                ),
            )
            seq.add_block(pp.make_delay(delay_te))

            grad_readout_x = pp.make_trapezoid(
                channel="x",
                amplitude=grad_readout_base.amplitude * dir_x,
                flat_time=grad_readout_base.flat_time,
                rise_time=grad_readout_base.rise_time,
                fall_time=grad_readout_base.fall_time,
                system=system,
            )
            grad_readout_y = pp.make_trapezoid(
                channel="y",
                amplitude=grad_readout_base.amplitude * dir_y,
                flat_time=grad_readout_base.flat_time,
                rise_time=grad_readout_base.rise_time,
                fall_time=grad_readout_base.fall_time,
                system=system,
            )
            grad_readout_z = pp.make_trapezoid(
                channel="z",
                amplitude=grad_readout_base.amplitude * dir_z,
                flat_time=grad_readout_base.flat_time,
                rise_time=grad_readout_base.rise_time,
                fall_time=grad_readout_base.fall_time,
                system=system,
            )

            if is_dummy:
                seq.add_block(grad_readout_x, grad_readout_y, grad_readout_z)
            else:
                seq.add_block(grad_readout_x, grad_readout_y, grad_readout_z, adc)

            seq.add_block(
                pp.make_trapezoid(
                    channel="x", area=grad_spoil_base.area * dir_x, system=system
                ),
                pp.make_trapezoid(
                    channel="y", area=grad_spoil_base.area * dir_y, system=system
                ),
                pp.make_trapezoid(
                    channel="z",
                    area=grad_spoil_base.area * dir_z + grad_slice_spoil.area,
                    system=system,
                ),
            )
            # Recovery between echoes: this is what makes each echo a different
            # effective TI (MPnRAGE).
            if shot < n_shots_segment - 1:
                seq.add_block(pp.make_delay(delay_ti_inter))

        seq.add_block(pp.make_delay(delay_recovery))

    timing_ok, error_report = seq.check_timing()
    if timing_ok:
        print("MPnRAGE timing check passed")
        print(f"  GRE block dur : {gre_block_dur * 1000:.1f} ms")
        print(f"  Echo spacing P: {(gre_block_dur + delay_ti_inter) * 1000:.1f} ms")
        print(
            f"  Readout train : "
            f"{n_readout * (gre_block_dur + delay_ti_inter) * 1000:.0f} ms"
        )
        scan_time_est = n_segments * (
            inversion_duration
            + delay_ti_start
            + n_readout * gre_block_dur
            + (n_readout - 1) * delay_ti_inter
            + recovery_delay
        )
        print(f"  Scan time est : {scan_time_est / 60:.1f} min")
    else:
        print(f"Timing FAILED: {error_report}")

    seq.set_definition("FOV", [fov, fov, slice_thickness])
    seq.set_definition("Name", filename.replace(".seq", ""))
    if write_seq:
        seq.write(filename)
    return seq, gre_block_dur, delay_ti_inter, recovery_delay

## Parameters

Single source of truth for the matrix size, the echo/segment layout and the
tissue parameters. Tissue parameters are exactly those defined in
`Modules/phantom.py` (CSF: PD 1.0, T1 4.0 s, T2 0.5 s; GM: PD 0.8, T1 1.3 s,
T2 0.08 s; WM: PD 0.7, T1 0.9 s, T2 0.07 s; T2* = 0.04 s everywhere).

In [ ]:
fov = 250e-3
n_x = 64  # phantom + recon matrix (n_x^3); readout samples/spoke so kmax matches
flip_angle_deg = 5.0
n_readout = 20  # echoes per inversion segment (each echo index = one image)
n_segments = 750  # inversion segments (n_segments * n_readout spokes in total)
n_dummy = 20
ti_start = 20e-3  # TI of the first echo; echo p fires ti_start + p * P later
ti_inter = 100e-3  # recovery delay between echoes (echo spacing P = TR + ti_inter)
n_channels = 8
spin_count = 64  # isochromats per voxel

tissue_params = {
    "CSF": dict(t1=4.0, t2=0.5, pd=1.0, pd_range=(0.9, 1.01)),
    "GM": dict(t1=1.3, t2=0.08, pd=0.8, pd_range=(0.75, 0.9)),
    "WM": dict(t1=0.9, t2=0.07, pd=0.7, pd_range=(0.1, 0.75)),
}
t2_star = 0.04
tissue_names = ("WM", "GM", "CSF")

## Phantom, tissue masks and coil sensitivities

BrainWeb phantom on an $n_x^3$ grid. The tissue masks are derived from the same
label volume and PD mapping as the phantom. The reconstruction operator uses k with
flipped sign, so the image is the phantom inverted through the origin; the image-space
masks are flipped accordingly.

In [ ]:
phantom = load_brain_phantom(
    raw_path="subject04_crisp_v.rawb",
    target_size=(n_x, n_x, n_x),
    device=device,
    permute_order=(2, 1, 0),
)

# Tissue masks on the same grid as the phantom (same mapping as phantom.py).
labels = np.fromfile("subject04_crisp_v.rawb", dtype=np.uint8).reshape(362, 434, 362)
pd_signal = np.zeros_like(labels, dtype=np.float32)
pd_signal[labels == 1] = 1.0
pd_signal[labels == 2] = 0.8
pd_signal[labels == 3] = 0.7
pd_map = (
    torch.nn.functional.interpolate(
        torch.from_numpy(pd_signal).to(device).view(1, 1, 362, 434, 362),
        size=(n_x, n_x, n_x),
        mode="trilinear",
    )
    .squeeze()
    .permute(2, 1, 0)
)
masks = {}
for tissue in tissue_names:
    pd_low, pd_high = tissue_params[tissue]["pd_range"]
    in_range = (pd_map > pd_low) & (pd_map < pd_high)
    masks[tissue] = in_range.to(torch.float32).cpu().numpy()
masks_img = {
    tissue: np.ascontiguousarray(np.flip(mask, (0, 1, 2)))
    for tissue, mask in masks.items()
}

scale = tuple(float(s) / fov for s in phantom.size)
coil_sim = cyclic_head_coil((n_channels, n_x, n_x, n_x), fov_scaling=scale).to(
    torch.complex64
)
phantom.coil_sens = coil_sim.to(device)
voxel_data = phantom.build()

## Null-point TIs

Build the sequence once (any TI) to get the exact timings: TR is the GRE block
duration and $P = TR + $ `ti_inter`. The timings do not depend on TI.

Symbols (see the derivation in the introduction): `lam` = $e^{-P/T_1}\cos\alpha$,
`kap` = $M_{z,ss}/M_0$, `e_tot` = $E_{tot}$, `z_star` = $z^*$, the cyclic state
before the inversion at the null.

In [ ]:
_, tr_echo, delay_ti_inter, recovery_delay = build_mpnrage(
    flip_angle_deg=flip_angle_deg, ti_start=100e-3
)
echo_spacing = tr_echo + delay_ti_inter
flip_angle_rad = flip_angle_deg * np.pi / 180


def null_ti(
    t1: float,
    echo_spacing: float,
    echo_tr: float = tr_echo,
    recovery_delay: float = recovery_delay,
    flip_angle_rad: float = flip_angle_rad,
    n_readout: int = n_readout,
) -> float:
    """Inversion time at which the first echo of a cyclic segment is nulled.

    The magnetization just before the inversion pulse is the fixed point of one
    segment (inversion, TI, ``n_readout`` echoes, recovery delay). At the null
    (Mz = 0 at the first excitation) that fixed point is independent of TI::

        z* = (1 - e_tot) + e_tot cos(a) kap (1 - lam^(N-1))
        e_tot = exp(-(TR + recovery_delay) / T1),  lam = exp(-P / T1) cos(a)
        kap = Mz_ss / M0 = (1 - E1) / (1 - E1 cos(a)),  E1 = exp(-P / T1)

    and the null condition ``1 - 2 E_TI + E_TI E_TD (1 - kap) = 0`` gives
    ``E_TI = 1 / (1 + z*)``.

    Parameters
    ----------
    t1 : float
        Longitudinal relaxation time in s.
    echo_spacing : float
        Echo spacing ``P = TR + ti_inter`` in s.
    echo_tr : float, optional
        Duration of one GRE block in s.
    recovery_delay : float, optional
        Delay after each echo train in s.
    flip_angle_rad : float, optional
        Excitation flip angle in rad.
    n_readout : int, optional
        Number of echoes per segment.

    Returns
    -------
    float
        Null inversion time in s.
    """
    e1 = np.exp(-echo_spacing / t1)
    lam = e1 * np.cos(flip_angle_rad)
    kap = (1 - e1) / (1 - e1 * np.cos(flip_angle_rad))
    e_tot = np.exp(-(echo_tr + recovery_delay) / t1)
    z_star = (1 - e_tot) + e_tot * np.cos(flip_angle_rad) * kap * (
        1 - lam ** (n_readout - 1)
    )
    return t1 * np.log(1.0 + z_star)


ti_null = {
    tissue: null_ti(params["t1"], echo_spacing)
    for tissue, params in tissue_params.items()
}
ti_effs = np.array([ti_start + p * echo_spacing for p in range(n_readout)])
null_frame = {
    tissue: int(np.argmin(np.abs(ti_effs - ti_null[tissue]))) for tissue in ti_null
}
for tissue in tissue_names:
    frame = null_frame[tissue]
    t1_ln2 = tissue_params[tissue]["t1"] * np.log(2)
    print(
        f"TI_null({tissue}) = {ti_null[tissue] * 1e3:7.1f} ms   frame {frame:2d} "
        f"(TI_eff = {ti_effs[frame] * 1e3:5.0f} ms)   "
        f"(T1*ln2 = {t1_ln2 * 1e3:7.1f} ms)"
    )

## Null check by simulation

A 4-segment, 32^3, single-tissue simulation (3 warm-up segments + 1 data segment,
so the data segment is on the cyclic state, which is needed for the long-T1 CSF).
The magnitude of the *first* acquired echo at `ti_null` is compared with a longer TI.

In [ ]:
def mini_first_echo(
    t1: float,
    t2: float,
    pd: float,
    ti: float,
    n_mini: int = 32,
    n_seg: int = 4,
    n_echoes: int = n_readout,
) -> float:
    """Simulate a homogeneous mini phantom and return the first echo magnitude.

    Parameters
    ----------
    t1 : float
        Longitudinal relaxation time in s.
    t2 : float
        Transverse relaxation time in s.
    pd : float
        Proton density.
    ti : float
        ``ti_start`` of the simulated sequence in s.
    n_mini : int, optional
        Matrix size of the mini phantom.
    n_seg : int, optional
        Number of simulated segments.
    n_echoes : int, optional
        Number of echoes per segment.

    Returns
    -------
    float
        Mean magnitude of the first spoke of the last segment.
    """
    shape = (n_mini, n_mini, n_mini)
    voxel_size = 250.0 / n_mini
    affine = torch.eye(4, device=device)
    for i in range(3):
        affine[i, i] = voxel_size
        affine[i, 3] = -voxel_size * (n_mini // 2)
    mini_phantom = mr0.VoxelGridPhantom(
        torch.full(shape, pd, device=device),
        torch.full(shape, t1, device=device),
        torch.full(shape, t2, device=device),
        torch.full(shape, t2_star, device=device),
        torch.zeros(shape, device=device),
        torch.zeros(shape, device=device),
        torch.ones((1,) + shape, device=device),
        torch.ones((1,) + shape, device=device),
        affine,
    )
    mini_phantom.coil_sens = torch.ones(
        (8,) + shape, dtype=torch.complex64, device=device
    )
    mini_data = mini_phantom.build()

    mini_filename = f"/tmp/mpnrage_check_{int(ti * 1000)}.seq"
    build_mpnrage(
        flip_angle_deg=flip_angle_deg,
        ti_start=ti,
        n_segments=n_seg,
        n_readout=n_echoes,
        write_seq=True,
        filename=mini_filename,
    )
    mini_seq = mr0.Sequence.import_file(mini_filename, exact_trajectories=False)
    torch.manual_seed(0)
    mini_signal = isochromat_sim_fast(
        mini_seq,
        mini_data,
        spin_count=spin_count,
        spin_dist="rand",
        print_progress=False,
    )
    mini_signal = mini_signal.permute(1, 0).contiguous()
    n_adc_per_spoke = mini_signal.shape[1] // (n_seg * n_echoes)
    last_segment = mini_signal[:, -n_echoes * n_adc_per_spoke :].sum(0)
    return last_segment[:n_adc_per_spoke].abs().mean().item()


for tissue in tissue_names:
    params = tissue_params[tissue]
    echo_at_null = mini_first_echo(
        params["t1"], params["t2"], params["pd"], ti_null[tissue]
    )
    echo_after_null = mini_first_echo(
        params["t1"], params["t2"], params["pd"], ti_null[tissue] + 300e-3
    )
    print(
        f"{tissue}: |1st echo| at TI_null = {echo_at_null:.5f}   "
        f"at TI_null+300ms = {echo_after_null:.5f}   "
        f"(ratio {echo_after_null / (echo_at_null + 1e-12):.1f}x)"
    )

## Simulate the one MPnRAGE scan

Remove stale caches first. The echo train spans `TI_eff` from `ti_start` up to
`ti_start + (n_readout - 1) * P`, which covers all three tissue nulls, so a single
simulation gives every frame.

In [ ]:
for stale_file in [
    "MPnRAGE_kooshball3D_signal.pt",
    "MPnRAGE_Kooshball_TI20ms.mr0cache.pt",
]:
    if os.path.exists(stale_file):
        os.remove(stale_file)

In [ ]:
seq_name = (
    f"MPnRAGE_Kooshball_TI{int(round(ti_start * 1000))}ms"
    f"_n{n_x}_R{n_readout}_S{n_segments}.seq"
)
build_mpnrage(
    fov=fov,
    n_x=n_x,
    flip_angle_deg=flip_angle_deg,
    n_readout=n_readout,
    n_segments=n_segments,
    n_dummy=n_dummy,
    ti_start=ti_start,
    ti_inter=ti_inter,
    write_seq=True,
    filename=seq_name,
)
seq_cache = seq_name.replace(".seq", ".mr0cache.pt")
try:
    mr_seq = torch.load(seq_cache, weights_only=False)
    print(f"loaded {seq_name} from cache")
except Exception:
    start_time = time.time()
    mr_seq = mr0.Sequence.import_file(seq_name, exact_trajectories=False)
    torch.save(mr_seq, seq_cache)
    print(f"imported {seq_name}: {time.time() - start_time:.0f} s (cached)")

n_adc = n_segments * n_readout * n_x  # dummies acquire no ADC samples
signal_file = "MPnRAGE_kooshball3D_signal.pt"
if os.path.exists(signal_file):
    signal = torch.load(signal_file)
    print(f"loaded saved signal {signal_file}  shape {tuple(signal.shape)}")
else:
    torch.manual_seed(0)
    start_time = time.time()
    signal = isochromat_sim_fast(
        mr_seq,
        voxel_data.cuda(),
        spin_count=spin_count,
        spin_dist="rand",
        print_progress=True,
        chunk_bytes=2**30,
    )
    print(f"simulation: {time.time() - start_time:.0f} s")
    torch.save(
        signal.permute((1, 0)).contiguous().to(torch.complex64).cpu(), signal_file
    )
# The simulator outputs (n_adc, n_channels); the recon pipeline wants (n_channels, n_adc).
if signal.shape[0] != n_channels:
    signal = signal.permute((1, 0))
signal = signal.to(device).contiguous().to(torch.complex64)
assert tuple(signal.shape) == (n_channels, n_adc), (
    tuple(signal.shape),
    (n_channels, n_adc),
)

## Helper functions for the adjoint-NUFFT previews

The ADC k-space is converted to rad/pixel, reconstructed per channel with a
density-compensated adjoint NUFFT, and the simulated coil maps are matched to the
orientation of the resulting images (best of the 48 axis permutations/flips).

In [ ]:
def kspace_to_rad(mr_seq: mr0.Sequence, n_x: int, fov: float) -> torch.Tensor:
    """Convert the ADC k-space (1/m) to rad/pixel of an ``n_x**3`` grid.

    Parameters
    ----------
    mr_seq : mr0.Sequence
        Sequence providing the ADC k-space.
    n_x : int
        Matrix size of the reconstruction grid.
    fov : float
        Field of view in m.

    Returns
    -------
    torch.Tensor
        k-space in rad/pixel, shape ``(3, n_adc_samples)``.
    """
    k = torch.as_tensor(mr_seq.get_kspace()).float()
    k = k.reshape(-1, k.shape[-1])[:, :3].T.contiguous()
    return k * (2 * np.pi * fov / n_x)


def nufft_recon(signal: torch.Tensor, k_rad: torch.Tensor, n_x: int) -> torch.Tensor:
    """Reconstruct each channel with a density-compensated adjoint NUFFT.

    Samples beyond the kmax of the grid are dropped (not clamped).

    Parameters
    ----------
    signal : torch.Tensor
        Multi-coil k-space data, shape ``(n_channels, n_samples)``.
    k_rad : torch.Tensor
        k-space trajectory in rad/pixel, shape ``(3, n_samples)``.
    n_x : int
        Matrix size of the reconstruction grid.

    Returns
    -------
    torch.Tensor
        Complex images of shape ``(n_channels, n_x, n_x, n_x)``.
    """
    k_rad = k_rad.to(signal.device)
    inside_kmax = k_rad.norm(dim=0) <= np.pi
    k_kept, signal_kept = k_rad[:, inside_kmax], signal[:, inside_kmax]
    # |k|^2 density compensation for radial sampling.
    density_comp = k_kept.norm(dim=0).clamp(min=np.pi / n_x) ** 2
    adjoint_nufft = tkbn.KbNufftAdjoint(
        im_size=(n_x, n_x, n_x), grid_size=(2 * n_x,) * 3
    ).to(signal.device)
    weighted = signal_kept * density_comp
    batch = weighted[None]
    return adjoint_nufft(batch, k_kept)[0]


def relative_profile(volume: np.ndarray) -> np.ndarray:
    """Normalise channel magnitudes by their root-sum-of-squares.

    Parameters
    ----------
    volume : np.ndarray
        Complex multi-coil volume, shape ``(n_channels, n_x, n_x, n_x)``.

    Returns
    -------
    np.ndarray
        Relative channel magnitudes of the same shape.
    """
    magnitude = np.abs(volume)
    rss = np.sqrt((magnitude**2).sum(0, keepdims=True))
    return magnitude / (rss + 1e-12)


def permute_and_flip(
    volume: np.ndarray, perm: Tuple[int, ...], flips: Tuple[bool, ...]
) -> np.ndarray:
    """Permute and flip the three spatial axes of a multi-coil volume.

    Parameters
    ----------
    volume : np.ndarray
        Volume of shape ``(n_channels, n_x, n_x, n_x)``.
    perm : Tuple[int, ...]
        Permutation of the spatial axes ``(0, 1, 2)``.
    flips : Tuple[bool, ...]
        Whether to flip each spatial axis after permuting.

    Returns
    -------
    np.ndarray
        Permuted and flipped volume.
    """
    permuted = volume.transpose(0, *[p + 1 for p in perm])
    flip_axes = tuple(axis + 1 for axis, flip in enumerate(flips) if flip)
    return np.flip(permuted, flip_axes) if flip_axes else permuted


def match_coil_orientation(
    coil: torch.Tensor, img: torch.Tensor, mask_frac: float = 0.3
) -> Tuple[np.ndarray, Tuple[int, ...], Tuple[bool, ...], float]:
    """Find the axis permutation/flip of the coil maps that matches the images.

    Compares ``|coil_c| / rss(coil)`` with ``|img_c| / rss(img)`` inside the head
    for all 48 axis permutations and flips of ``coil``.

    Parameters
    ----------
    coil : torch.Tensor
        Coil sensitivities, shape ``(n_channels, n_x, n_x, n_x)``.
    img : torch.Tensor
        Raw NUFFT images, shape ``(n_channels, n_x, n_x, n_x)``.
    mask_frac : float, optional
        Fraction of the maximum root-sum-of-squares image defining the head mask.

    Returns
    -------
    coil_matched : np.ndarray
        Coil maps in the orientation of the images.
    perm : Tuple[int, ...]
        Best axis permutation.
    flips : Tuple[bool, ...]
        Best axis flips.
    score : float
        Correlation of the best match (close to 1 for a good match).
    """
    coil, img = np.asarray(coil), np.asarray(img)
    rss = np.sqrt((np.abs(img) ** 2).sum(0))
    mask = rss > mask_frac * rss.max()
    ref = relative_profile(img)[:, mask].ravel()
    ref = ref - ref.mean()

    best_score, best_perm, best_flips = -2.0, None, None
    for perm in itertools.permutations((0, 1, 2)):
        for flips in itertools.product([False, True], repeat=3):
            candidate = permute_and_flip(coil, perm, flips)
            profile = relative_profile(candidate)[:, mask].ravel()
            profile = profile - profile.mean()
            norm_product = np.linalg.norm(ref) * np.linalg.norm(profile) + 1e-12
            score = float(ref @ profile / norm_product)
            if score > best_score:
                best_score, best_perm, best_flips = score, perm, flips

    coil_matched = np.ascontiguousarray(permute_and_flip(coil, best_perm, best_flips))
    return coil_matched, best_perm, best_flips, best_score


def adc_sample_indices(echo_indices: Iterable[int]) -> torch.Tensor:
    """Return the flat ADC sample indices of the given echo indices.

    Dummy shots acquire no ADC samples, so echo ``p`` of segment ``s`` is ADC
    block ``s * n_readout + p`` and covers ``n_x`` consecutive samples. The
    indices are ordered echo by echo, and by segment within each echo.

    Parameters
    ----------
    echo_indices : Iterable[int]
        Echo indices (0 ... ``n_readout - 1``) to collect across all segments.

    Returns
    -------
    torch.Tensor
        Flat ADC sample indices of length ``len(echo_indices) * n_segments * n_x``.
    """
    blocks = [s * n_readout + p for p in echo_indices for s in range(n_segments)]
    return torch.cat([torch.arange(b * n_x, (b + 1) * n_x) for b in blocks])

## One image per echo index

Echo $p$ of every segment fired `TI_eff(p)` after that segment's inversion, so each
frame is the brain at a single effective TI. Frame $p$ uses the $p$-th acquired echo
of all `n_segments` segments (`n_segments` spokes per frame), reconstructed with a
density-compensated adjoint NUFFT and a sensitivity-weighted coil combination.

In [ ]:
k_full = kspace_to_rad(mr_seq, n_x, fov)
assert k_full.shape[1] == signal.shape[1], (k_full.shape, signal.shape)

coil_recon = None
frame_imgs = {}
for p in range(n_readout):
    idx = adc_sample_indices([p])
    img = nufft_recon(signal[:, idx], k_full[:, idx], n_x)  # raw NUFFT orientation
    if coil_recon is None:
        coil_np, perm, flips, score = match_coil_orientation(
            coil_sim.cpu().numpy(), img.cpu().numpy()
        )
        print(f"coil orientation: perm={perm} flips={flips} score={score:.3f}")
        coil_recon = torch.from_numpy(coil_np).to(img.device)
    sens_power = coil_recon.abs().square().sum(0)
    combined = (coil_recon.conj() * img).sum(0) / sens_power.clamp(
        min=1e-2 * sens_power.max()
    )
    frame_imgs[p] = combined.abs().cpu().numpy()
    print(f"frame {p:2d}  TI_eff = {ti_effs[p] * 1e3:7.1f} ms")

## Contrast evolution

One brain per echo index; the frames closest to a tissue's null TI are tagged.

In [ ]:
mid = n_x // 2
fig, axes = plt.subplots(4, 5, figsize=(16, 13))
for p, ax in enumerate(axes.ravel()):
    ax.imshow(np.rot90(frame_imgs[p][:, :, mid].T, 2), cmap="gray")
    tag = "   <- " + ",".join(t for t in tissue_names if null_frame[t] == p)
    ax.set_title(f"frame {p}\nTI = {ti_effs[p] * 1e3:.0f} ms{tag}", fontsize=9)
    ax.axis("off")
plt.suptitle("MPnRAGE - one scan, one image per echo index (NUFFT-adjoint previews)")
plt.tight_layout()
plt.show()

## Per-tissue contrast vs effective TI

The nulls are the minima of the mask-mean signal.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for tissue in tissue_names:
    mask = masks_img[tissue] > 0.5
    mask_means = [frame_imgs[p][mask].mean() for p in range(n_readout)]
    ax.plot(
        ti_effs * 1e3,
        mask_means,
        "o-",
        markersize=4,
        label=f"{tissue} (null {ti_null[tissue] * 1e3:.0f} ms)",
    )
    ax.axvline(ti_null[tissue] * 1e3, ls=":", alpha=0.4, color=ax.lines[-1].get_color())
ax.set_xlabel("Effective TI (ms)")
ax.set_ylabel("Mask-mean signal (reconstructed frame)")
ax.set_title("MPnRAGE magnetization recovery - per-tissue contrast vs TI")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## k-space trajectory for the iterative reconstruction

Read the ADC trajectory of all spokes (every echo index, every segment) from the
sequence file and normalise it to $[-0.5, 0.5]$.

In [ ]:
seq = pp.Sequence()
seq.read(seq_name)
kspace = seq.calculate_kspace()
k_traj_adc = kspace[0]
k_traj_adc = k_traj_adc[:3, :]
k_traj_radius = np.sqrt(np.sum(k_traj_adc**2, axis=0, keepdims=True))
k_traj_adc = k_traj_adc / k_traj_radius.max() * 0.5
k_traj_adc = torch.tensor(k_traj_adc, dtype=torch.float32)

## Linear operators

Complex tensors are handled as flattened real vectors (`view_as_real`), so the
operators act on real vectors of twice the length. `ChannelOperator` maps an image
to multi-coil images, `NonuniformFourierTransformOperator` maps multi-coil images
to non-Cartesian k-space samples, and `IdentityOperator` is used by ADMM.

In [ ]:
class ChannelOperator(LinearOperator):
    """Coil sensitivity operator: image to multi-coil images.

    Parameters
    ----------
    coil_sensitivities : torch.Tensor
        Complex coil sensitivities of shape ``(n_channels, *data_shape)``.
    data_shape : Tuple[int, ...]
        Spatial shape of the image.
    device : Optional[torch.device], optional
        Device attribute of the operator.
    """

    def __init__(
        self,
        coil_sensitivities: torch.Tensor,
        data_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ) -> None:
        n_channels = coil_sensitivities.shape[0]

        self.coil_sensitivities = coil_sensitivities

        self.forward_shape = (1, *data_shape)
        self.adjoint_shape = (n_channels, *data_shape)
        self.shape = (
            int(2 * torch.prod(torch.tensor(self.adjoint_shape))),
            int(2 * torch.prod(torch.tensor(self.forward_shape))),
        )

        self.dtype = torch.float32
        self.device = device

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the operator: image to multi-coil images.

        Parameters
        ----------
        x : torch.Tensor
            Real-valued flattened image.

        Returns
        -------
        torch.Tensor
            Real-valued flattened multi-coil images.
        """
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.forward_shape)
        y = self.coil_sensitivities * x
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the adjoint: multi-coil images to coil-combined image.

        Parameters
        ----------
        x : torch.Tensor
            Real-valued flattened multi-coil images.

        Returns
        -------
        torch.Tensor
            Real-valued flattened coil-combined image.
        """
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.adjoint_shape)
        y = torch.sum(torch.conj(self.coil_sensitivities) * x, dim=0, keepdim=True)
        return torch.view_as_real(y.reshape(-1)).reshape(-1)


class NonuniformFourierTransformOperator(LinearOperator):
    """Non-Cartesian Fourier operator: multi-coil 3D images to k-space samples.

    Parameters
    ----------
    k : torch.Tensor
        k-space trajectory in cycles/voxel, shape ``(3, n_samples)``. The sign is
        flipped internally to match the NUFFT convention.
    input_shape : Tuple[int, ...]
        Shape ``(n_channels, n_x, n_y, n_z)`` of the multi-coil images.
    device : Optional[torch.device], optional
        Device on which the NUFFT objects are placed.
    """

    def __init__(
        self,
        k: torch.Tensor,
        input_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ) -> None:
        _, n_samples = k.shape
        n_channels, n_x, n_y, n_z = input_shape
        self.n_modes = (n_x, n_y, n_z)
        self.k = (-k).unsqueeze(-1)  # (3, n_samples, 1)
        self.forward_shape = (n_channels, n_x, n_y, n_z, 1)
        self.adjoint_shape = (n_channels, n_samples)
        self.shape = (
            int(2 * torch.prod(torch.tensor(self.adjoint_shape))),
            int(2 * torch.prod(torch.tensor(self.forward_shape))),
        )
        self.nufft_ob = KbNufft(im_size=self.n_modes).to(device)
        self.adj_ob = KbNufftAdjoint(im_size=self.n_modes).to(device)
        self.dtype = torch.float32
        self.device = device

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the forward NUFFT.

        Parameters
        ----------
        x : torch.Tensor
            Real-valued flattened multi-coil images.

        Returns
        -------
        torch.Tensor
            Real-valued flattened k-space samples.
        """
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.forward_shape)
        y = nonuniform_fourier_transform_forward(self.k, x, nufft_ob=self.nufft_ob)
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the adjoint NUFFT.

        Parameters
        ----------
        x : torch.Tensor
            Real-valued flattened k-space samples.

        Returns
        -------
        torch.Tensor
            Real-valued flattened multi-coil images.
        """
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.adjoint_shape)
        y = nonuniform_fourier_transform_adjoint(
            self.k, x, self.n_modes, adj_ob=self.adj_ob
        )
        return torch.view_as_real(y.reshape(-1)).reshape(-1)


class IdentityOperator(LinearOperator):
    """Identity operator on real-valued views of complex images.

    Parameters
    ----------
    data_shape : Tuple[int, ...]
        Shape of the (complex) image.
    device : Optional[torch.device], optional
        Device attribute of the operator.
    """

    def __init__(
        self,
        data_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ) -> None:
        self.shape = (
            int(2 * torch.prod(torch.tensor(data_shape))),
            int(2 * torch.prod(torch.tensor(data_shape))),
        )

        self.dtype = torch.float32
        self.device = device

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        """Return ``x`` unchanged.

        Parameters
        ----------
        x : torch.Tensor
            Real-valued flattened image.

        Returns
        -------
        torch.Tensor
            The input.
        """
        return x

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        """Return ``x`` unchanged (the operator is self-adjoint).

        Parameters
        ----------
        x : torch.Tensor
            Real-valued flattened image.

        Returns
        -------
        torch.Tensor
            The input.
        """
        return x

## DeepADMM

Solves $\min_x \tfrac{1}{2 n}\|Ax - d\|_2^2 + \rho\, g(x)$ by ADMM with a learned
regularizer $g$ ($n$ = number of complex k-space samples). The $x$-update is a few CG
iterations on $(Q + \rho I)\,x = b + \rho\,(z - u)$. $Q$ and $b$ must be the
**normalised** data term, $Q = A^H A / n$ and $b = A^H d / n$.

In [ ]:
class DeepADMM(nn.Module):
    """ADMM solver with a learned 3D regularizer.

    Solves ``min_x 1/(2 n_samples) ||A x - d||^2 + rho * g(x)`` with ``g`` a
    learned regularizer.

    Parameters
    ----------
    shape : Tuple[int, ...]
        Shape of the image the regularizer and identity operator act on.
    rho_init : float, optional
        Initial value of the ADMM penalty parameter ``rho``.
    maxiter_admm : int, optional
        Default number of ADMM iterations.
    maxiter_cg : int, optional
        Default number of CG iterations per ADMM ``x``-update.
    kernel_size : Tuple[int, ...], optional
        Convolution kernel size of the regularizer.
    features : int, optional
        Number of feature channels of the regularizer.
    num_of_resblocks : int, optional
        Number of residual blocks of the regularizer.
    scale_factor : int, optional
        Scale factor of the regularizer.
    checkpoint_iterations : bool, optional
        Use gradient checkpointing for each ADMM iteration while training.

    Notes
    -----
    ``Q`` and ``b`` passed to :meth:`forward` must be the *normalised* data term,
    ``Q = A^H A / n_samples`` and ``b = A^H d / n_samples``. For this dataset
    ``lambda_max(Q)`` is about 1.2e-3, so ``rho`` of 1e-4 to 1e-3 is on the same
    scale and the regularizer actually has an effect.
    """

    def __init__(
        self,
        shape: Tuple[int, ...],
        rho_init: float = 3e-4,
        maxiter_admm: int = 1,
        maxiter_cg: int = 20,
        kernel_size: Tuple[int, ...] = (3, 3, 3),
        features: int = 16,
        num_of_resblocks: int = 4,
        scale_factor: int = 1,
        checkpoint_iterations: bool = True,
    ) -> None:
        super().__init__()
        self.rho = nn.Parameter(torch.tensor(rho_init, dtype=torch.float32))
        self.regularizer = Regularizer(
            shape,
            kernel_size=kernel_size,
            features=features,
            num_of_resblocks=num_of_resblocks,
            scale_factor=scale_factor,
        )
        self.I = IdentityOperator(shape)
        self.maxiter_admm = maxiter_admm
        self.maxiter_cg = maxiter_cg
        self.checkpoint_iterations = checkpoint_iterations

    def _step(
        self,
        x: torch.Tensor,
        z: torch.Tensor,
        u: torch.Tensor,
        Q: LinearOperator,
        b: torch.Tensor,
        maxiter_cg: int,
        disable_progressbar: bool = True,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Perform one ADMM iteration.

        Parameters
        ----------
        x : torch.Tensor
            Current image estimate (real-valued view).
        z : torch.Tensor
            Current auxiliary variable.
        u : torch.Tensor
            Current scaled dual variable.
        Q : LinearOperator
            Normalised normal operator ``A^H A / n_samples``.
        b : torch.Tensor
            Normalised right-hand side ``A^H d / n_samples``.
        maxiter_cg : int
            Number of CG iterations of the ``x``-update.
        disable_progressbar : bool, optional
            Disable the CG progress bar.

        Returns
        -------
        Tuple[torch.Tensor, torch.Tensor, torch.Tensor]
            Updated ``x``, ``z`` and ``u``.
        """
        x = conjugate_gradient(
            Q + self.rho * self.I,
            b=b + self.rho * (z - u),
            x=x,
            maxiter=maxiter_cg,
            disable_progressbar=disable_progressbar,
        )
        z = self.regularizer(x + u)
        u = u + x - z
        return x, z, u

    def forward(
        self,
        Q: LinearOperator,
        b: torch.Tensor,
        x: torch.Tensor,
        z: Optional[torch.Tensor] = None,
        maxiter_admm: Optional[int] = None,
        maxiter_cg: Optional[int] = None,
        disable_progressbar: bool = True,
    ) -> torch.Tensor:
        """Run the unrolled ADMM iterations, warm-started from ``x``.

        Parameters
        ----------
        Q : LinearOperator
            Normalised normal operator ``A^H A / n_samples``.
        b : torch.Tensor
            Normalised right-hand side ``A^H d / n_samples``.
        x : torch.Tensor
            Warm start, e.g. the CG solution (real-valued view).
        z : Optional[torch.Tensor], optional
            Initial auxiliary variable; defaults to ``regularizer(x)``.
        maxiter_admm : Optional[int], optional
            Number of ADMM iterations; defaults to the constructor value.
        maxiter_cg : Optional[int], optional
            Number of CG iterations per ``x``-update; defaults to the constructor
            value.
        disable_progressbar : bool, optional
            Disable the CG progress bar.

        Returns
        -------
        torch.Tensor
            Reconstructed image (real-valued view).
        """
        n_admm = self.maxiter_admm if maxiter_admm is None else maxiter_admm
        n_cg = self.maxiter_cg if maxiter_cg is None else maxiter_cg
        if z is None:
            z = self.regularizer(x)
        u = torch.zeros(self.I.shape[0], dtype=x.dtype)
        use_checkpoint = (
            self.checkpoint_iterations and self.training and torch.is_grad_enabled()
        )
        for _ in range(n_admm):
            if use_checkpoint:
                x, z, u = checkpoint(
                    self._step,
                    x,
                    z,
                    u,
                    Q,
                    b,
                    n_cg,
                    disable_progressbar,
                    use_reentrant=False,
                )
            else:
                x, z, u = self._step(x, z, u, Q, b, n_cg, disable_progressbar)
        return x

## CG-SENSE at each tissue's null TI

Notation: $C$ is the coil-sensitivity operator, $F$ the NUFFT, $A = F C$ the
encoding operator, $Q = A^H A$ and $b = A^H d$. `cg_sense` solves $Q x = b$ for the
ADC samples of a chosen echo frame, starting from zero.

In [ ]:
C = ChannelOperator(coil_sim, (n_x, n_x, n_x))
cg_iters = 30


def cg_sense(
    sample_idx: torch.Tensor, maxiter: int = cg_iters, disable_progressbar: bool = True
) -> torch.Tensor:
    """Iterative NUFFT-SENSE (CG) reconstruction from a subset of ADC samples.

    Parameters
    ----------
    sample_idx : torch.Tensor
        Flat ADC sample indices to reconstruct from.
    maxiter : int, optional
        Number of CG iterations.
    disable_progressbar : bool, optional
        Disable the CG progress bar.

    Returns
    -------
    torch.Tensor
        Complex image of shape ``(n_x, n_x, n_x)``.
    """
    F_p = NonuniformFourierTransformOperator(
        k_traj_adc[:, sample_idx], (n_channels, n_x, n_x, n_x)
    )
    d_p = signal[:, sample_idx].contiguous().view(torch.float32).ravel()
    Q_p = C.H @ F_p.H @ F_p @ C
    b_p = C.H @ F_p.H @ d_p
    x_init = torch.zeros(n_x**3, dtype=torch.complex64).view(torch.float32)
    x_recon = conjugate_gradient(
        Q_p, b_p, x_init, maxiter=maxiter, disable_progressbar=disable_progressbar
    )
    return x_recon.view(torch.complex64).reshape(n_x, n_x, n_x)

In [ ]:
half_width = 0  # 0 = only the null echo; 1 = null echo +/- 1 (3 echoes), etc.

null_vols = {}
for tissue in tissue_names:
    null_echo = null_frame[tissue]
    echo_indices = range(
        max(0, null_echo - half_width), min(n_readout, null_echo + half_width + 1)
    )
    idx = adc_sample_indices(echo_indices)
    recon = cg_sense(idx, cg_iters, disable_progressbar=False)
    null_vols[tissue] = torch.abs(recon).cpu().numpy()
    print(
        f"{tissue} nulled: frame {null_echo}, "
        f"TI_eff = {ti_effs[null_echo] * 1e3:.0f} ms, {len(idx) // n_x} spokes"
    )

Switch to the static backend for the remaining figures.

In [ ]:
%matplotlib inline
plt.close("all")

In [ ]:
# Rows = nulled tissue, columns = 3 orthogonal planes.
fig, axes = plt.subplots(3, 3, figsize=(5, 5))
for row, tissue in enumerate(tissue_names):
    vol = null_vols[tissue]
    vmax = np.percentile(vol, 99.5)  # one display scale per row
    views = [
        np.rot90(vol[n_x // 2, :, :].T, 2),
        np.rot90(vol[:, n_x // 2, :].T, 2),
        np.rot90(vol[:, :, n_x // 2].T, 2),
    ]
    for col, (view, plane) in enumerate(zip(views, ["X-slice", "Y-slice", "Z-slice"])):
        axes[row, col].imshow(view, cmap="gray", vmin=0, vmax=vmax)
        axes[row, col].set_title(f"{tissue} nulled - {plane}", fontsize=10)
        axes[row, col].axis("off")
    axes[row, 0].text(
        -0.05,
        0.5,
        f"{tissue} nulled\nTI = {ti_effs[null_frame[tissue]] * 1e3:.0f} ms",
        transform=axes[row, 0].transAxes,
        ha="right",
        va="center",
        fontsize=11,
    )

plt.suptitle("CG-SENSE reconstruction at each tissue's null TI")
plt.tight_layout()
plt.show()

## Why the original DeepADMM looked identical to CG - and the fix

**Diagnosis** (all measured on this dataset):

1. **The regularizer's weight was ~4 orders of magnitude too small.** The ADMM x-update
   solves `(Q + rho*I) x = b + rho*(z-u)` with the *unnormalized* Gram matrix
   `Q = A^H A`. Here `lambda_max(Q) ~= 450` (750 spokes) while `rho = 0.1`, so the
   learned regularizer contributed only `rho/lambda_max ~= 2e-4` of the data term: the
   ADMM fixed point was numerically the plain-CG solution, whatever the network learned.
2. **Learnable `rho` drifted downward** (old checkpoint: 0.1 -> 0.087; with the
   normalized data term it collapses to ~1e-10 in a single step), because the
   consistency-only loss prefers a weaker regularizer.
3. **The full 750-spoke frame is over-determined** (768k real measurements vs 524k real
   unknowns, ratio 1.46): on noise-free over-determined data the least-squares solution
   is (nearly) unique, so *no* regularizer can change the answer - the old evaluation
   left no room for an "improvement".
4. **The network was trained in the broken regime** (its effect was ~0 during the
   unroll), so it never learned a useful prior; given a meaningful rho it actually
   degraded the image.
5. Bonus: the 30-iteration CG baseline is itself under-converged - radial `A^H A` is
   ill-conditioned, and its error is ~3x that of a converged CG on the full frame.

**Fix** (verified below):

- Normalize the data term: `Q = A^H A / n_samples`, `b = A^H d / n_samples` ->
  `lambda_max ~= 1.2e-3` (independent of the number of spokes); `rho = 3e-4` is then
  on the same scale, and is **fixed** as a hyperparameter.
- Pre-train the regularizer as an aliasing denoiser (~45 min incl. pair building), so it
  actually regularizes once ADMM gives it real weight.
- Evaluate at under-determined sampling (150/60 spokes), where CG carries aliasing and
  the learned prior wins; the full-frame comparison is kept as a sanity check.

## DeepADMM training

### Full-frame references

Pseudo ground truth for the denoiser: a well-converged CG (120 iterations) on the
full null frame of each tissue.

In [ ]:
plt.close("all")
null_ps = [null_frame[tissue] for tissue in tissue_names]  # e.g. [5, 7, 14]
null_frame_idx = {p: adc_sample_indices([p]) for p in null_ps}


def gather_spokes(sample_idx: torch.Tensor, spokes: np.ndarray) -> torch.Tensor:
    """Select whole spokes from a flat ADC sample index.

    Parameters
    ----------
    sample_idx : torch.Tensor
        Flat ADC sample indices, ``n_x`` consecutive samples per spoke.
    spokes : np.ndarray
        Indices of the spokes to keep.

    Returns
    -------
    torch.Tensor
        Flat ADC sample indices of the selected spokes.
    """
    return torch.cat([sample_idx[s * n_x : (s + 1) * n_x] for s in spokes.tolist()])


print("computing full-frame references (CG 120)...")
ref_vols = {}
for tissue in tissue_names:
    idx = null_frame_idx[null_frame[tissue]]
    ref_vols[tissue] = cg_sense(idx, maxiter=120)
    print(f"  {tissue} done")

### Aliased training pairs

CG reconstructions of random under-sampled subsets of each null frame (150, 100 and
60 spokes, 5 repetitions each) are paired with the full-frame reference.

In [ ]:
sub_spokes = (150, 100, 60)
reps = 5
aliased_list, reference_list = [], []
for tissue in tqdm(tissue_names, desc="pairs (per tissue)"):
    idx_all = null_frame_idx[null_frame[tissue]]
    n_spokes_frame = len(idx_all) // n_x
    for n_sub in sub_spokes:
        for rep in range(reps):
            subset = np.random.RandomState(7000 + n_sub * 10 + rep).permutation(
                n_spokes_frame
            )[:n_sub]
            idx = gather_spokes(idx_all, subset)
            aliased_list.append(cg_sense(idx, cg_iters))
            reference_list.append(ref_vols[tissue])
aliased = torch.stack(aliased_list)[:, None]  # (N, 1, n_x, n_x, n_x)
reference = torch.stack(reference_list)[:, None]  # (N, 1, n_x, n_x, n_x)

### Pre-train the regularizer as an aliasing denoiser

Loss = MSE(denoise aliased) + 0.5 * MSE(identity on clean images); the second term
makes the network behave like a genuine proximal operator (identity on already
clean images).

In [ ]:
denoiser = ResNet(
    contrasts=1,
    features=16,
    num_of_resblocks=4,
    scale_factor=1,
    kernel_size=(3, 3, 3),
    device=device,
    dtype=torch.complex64,
)
optimizer = torch.optim.Adam(denoiser.parameters(), lr=1e-3)
batch_size = 6
losses = []
for phase, (n_steps, lr) in enumerate([(1200, 1e-3), (4000, 3e-4)]):
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr
    scheduler = (
        torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_steps)
        if phase
        else None
    )
    for step in tqdm(range(n_steps), desc=f"denoiser pretrain phase {phase + 1}"):
        batch_idx = torch.randperm(len(aliased))[:batch_size]
        target = reference[batch_idx]
        denoise_loss = ((denoiser(aliased[batch_idx]) - target).abs() ** 2).mean()
        identity_loss = ((denoiser(target) - target).abs() ** 2).mean()
        loss = denoise_loss + 0.5 * identity_loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        losses.append(loss.item())
print(f"final pretrain loss {losses[-1]:.3e}")

In [ ]:
smooth_window = 10
plt.figure(figsize=(6, 3))
plt.plot(losses, alpha=0.4)
plt.plot(
    np.arange(smooth_window - 1, len(losses)),
    np.convolve(losses, np.ones(smooth_window) / smooth_window, "valid"),
)
plt.xlabel("step")
plt.ylabel("denoiser pretrain loss")
plt.show()

### Assemble the DeepADMM model

The pre-trained denoiser becomes the ADMM regularizer. `rho` is a fixed
hyperparameter (a learnable `rho` collapses under the consistency-only loss).

In [ ]:
torch.manual_seed(0)
model = DeepADMM((n_x, n_x, n_x, 1), rho_init=3e-4, maxiter_admm=1, maxiter_cg=20)
model.regularizer.regularizer.load_state_dict(denoiser.state_dict())
model.rho.requires_grad_(False)
torch.save(model.state_dict(), "deepadmm_nullframes.pt")
print("model saved to deepadmm_nullframes.pt")

## Evaluation at several sampling levels

750 spokes (`n_segments`) is the full null frame (over-determined); 150 and 60 spokes
are under-determined. The error is the mean absolute difference to the full-frame
reference inside the head, relative to the mean reference magnitude. DeepADMM is
warm-started from the 30-iteration CG result and uses the normalised data term.

In [ ]:
model.eval()


def recon_admm(
    sample_idx: torch.Tensor,
    x_init: torch.Tensor,
    maxiter_admm: int = 1,
    maxiter_cg: int = 20,
) -> torch.Tensor:
    """DeepADMM reconstruction: normalised data term and learned regularizer.

    Parameters
    ----------
    sample_idx : torch.Tensor
        Flat ADC sample indices to reconstruct from.
    x_init : torch.Tensor
        Warm-start image of shape ``(n_x, n_x, n_x)`` (complex).
    maxiter_admm : int, optional
        Number of ADMM iterations.
    maxiter_cg : int, optional
        Number of CG iterations per ADMM ``x``-update.

    Returns
    -------
    torch.Tensor
        Complex image of shape ``(n_x, n_x, n_x)``.
    """
    F_p = NonuniformFourierTransformOperator(
        k_traj_adc[:, sample_idx], (n_channels, n_x, n_x, n_x)
    )
    d_p = signal[:, sample_idx].contiguous().view(torch.float32).ravel()
    n_samples = d_p.numel() // 2
    Q_p = (C.H @ F_p.H @ F_p @ C) * (1.0 / n_samples)  # normalised, as in training
    b_p = (C.H @ F_p.H @ d_p) * (1.0 / n_samples)
    x_init_flat = x_init.reshape(-1).view(torch.float32).reshape(-1)
    with torch.no_grad():
        x = model(
            Q_p, b_p, x_init_flat, maxiter_admm=maxiter_admm, maxiter_cg=maxiter_cg
        )
    return x.view(torch.complex64).reshape(n_x, n_x, n_x)


def relative_error(
    x: torch.Tensor, ref: torch.Tensor, head_mask: torch.Tensor, ref_mean: float
) -> float:
    """Mean absolute error inside the head relative to the mean reference.

    Parameters
    ----------
    x : torch.Tensor
        Reconstructed complex volume.
    ref : torch.Tensor
        Reference complex volume of shape ``(n_x, n_x, n_x)``.
    head_mask : torch.Tensor
        Boolean mask of the voxels to evaluate.
    ref_mean : float
        Mean reference magnitude inside the mask.

    Returns
    -------
    float
        Relative error.
    """
    return float(torch.abs(x.reshape(n_x, n_x, n_x) - ref)[head_mask].mean() / ref_mean)


eval_spokes = [n_segments, 150, 60]
results = {}
for tissue in tissue_names:
    idx_full = null_frame_idx[null_frame[tissue]]
    n_spokes_frame = len(idx_full) // n_x
    ref = ref_vols[tissue]
    ref_magnitude = ref.abs()
    head_mask = ref_magnitude > 0.2 * ref_magnitude.max()
    ref_mean = ref_magnitude[head_mask].mean().item()

    print(f"{tissue} nulled (reference: full-frame CG120)")
    for n_sub in eval_spokes:
        if n_sub < n_spokes_frame:
            subset = np.random.RandomState(9000 + n_sub).permutation(n_spokes_frame)[
                :n_sub
            ]
            idx = gather_spokes(idx_full, subset)
        else:
            idx = idx_full
        x_cg = cg_sense(idx, maxiter=cg_iters)
        x_cg_long = cg_sense(idx, maxiter=200)
        x_admm = recon_admm(idx, x_cg)
        results[(tissue, n_sub)] = (x_cg, x_cg_long, x_admm)
        print(
            f"  {n_sub:4d} spokes:  "
            f"CG({cg_iters})={relative_error(x_cg, ref, head_mask, ref_mean):.4f}   "
            f"CG(200)={relative_error(x_cg_long, ref, head_mask, ref_mean):.4f}   "
            f"DeepADMM={relative_error(x_admm, ref, head_mask, ref_mean):.4f}"
        )

null_admm = {
    tissue: results[(tissue, n_segments)][2].abs().cpu().numpy()
    for tissue in tissue_names
}

## CG-SENSE vs DeepADMM at the null TIs

In [ ]:
for method, vols in [("CG-SENSE", null_vols), ("DeepADMM", null_admm)]:
    fig, axes = plt.subplots(3, 3, figsize=(5, 5))
    for row, tissue in enumerate(tissue_names):
        vol = vols[tissue]
        vmax = np.percentile(vol, 99.5)
        views = [
            np.rot90(vol[n_x // 2].T, 2),
            np.rot90(vol[:, n_x // 2].T, 2),
            np.rot90(vol[:, :, n_x // 2].T, 2),
        ]
        for col, view in enumerate(views):
            axes[row, col].imshow(view, cmap="gray", vmin=0, vmax=vmax)
            axes[row, col].axis("off")
        axes[row, 0].set_title(
            f"{tissue} nulled, TI = {ti_effs[null_frame[tissue]] * 1e3:.0f} ms",
            fontsize=9,
            loc="left",
        )
    plt.suptitle(f"{method} - null-TI frames")
    plt.tight_layout()
    plt.show()